# BAN404 — General Python Cheatsheet
### Data exploration, manipulation, and visualisation
*Use together with the Model Recipe Book. This sheet covers everything before and after the models.*


## 0 · Standard Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Optional but useful
from scipy import stats
import statsmodels.api as sm

# Consistent plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100


---
## 1 · Loading and Inspecting Data


In [ ]:
# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv("data.csv")
df = pd.read_csv("data.csv", sep=';')        # semicolon-separated (European format)
df = pd.read_csv("data.csv", index_col=0)    # first column = row index

# ── First look ───────────────────────────────────────────────────────────────
print(df.shape)           # (rows, columns)
print(df.dtypes)          # data types of each column
df.head()                 # first 5 rows
df.tail()                 # last 5 rows
df.sample(5)              # 5 random rows

# ── Column names ─────────────────────────────────────────────────────────────
print(df.columns.tolist())

# ── Summary statistics ────────────────────────────────────────────────────────
df.describe()             # mean, std, min, quartiles, max for numeric cols
df.describe(include='all')  # includes categorical columns too


In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
df.isnull().sum()                    # count missing per column
df.isnull().sum() / len(df) * 100    # missing percentage per column
df.dropna()                          # drop rows with any missing
df.fillna(df.median(numeric_only=True))  # fill numeric NaN with median
df['col'].fillna('Unknown')          # fill categorical NaN

# ── Unique values ─────────────────────────────────────────────────────────────
df['col'].nunique()           # number of unique values
df['col'].value_counts()      # frequency table
df['col'].value_counts(normalize=True)  # proportions (sum to 1)


---
## 2 · Data Types and Recoding


In [ ]:
# ── Check and convert types ───────────────────────────────────────────────────
df['col'] = df['col'].astype(int)
df['col'] = df['col'].astype(float)
df['col'] = df['col'].astype(str)

# ── Recode binary string → 0/1 ───────────────────────────────────────────────
df['attrition'] = df['attrition'].map({'Yes': 1, 'No': 0})
df['overtime']  = df['overtime'].map({'Yes': 1, 'No': 0})

# ── Recode with conditions ───────────────────────────────────────────────────
df['high_income'] = (df['monthly_income'] > 5000).astype(int)
df['age_group']   = pd.cut(df['age'], bins=[0, 30, 45, 100],
                            labels=['Young', 'Mid', 'Senior'])

# ── Dummy encoding (for nominal categorical variables) ───────────────────────
df = pd.get_dummies(df, columns=['department'], drop_first=True)
# drop_first=True avoids multicollinearity (reference category is dropped)

# ── Label encoding (for ordinal variables — keeps order) ────────────────────
# e.g. education: 1=Below College, 2=College, ..., 5=Doctor → keep as int
# No action needed if already coded as integers

# ── Rename columns ───────────────────────────────────────────────────────────
df = df.rename(columns={'old_name': 'new_name'})


---
## 3 · Subsetting and Filtering


In [ ]:
# ── Select columns ───────────────────────────────────────────────────────────
df[['col1', 'col2']]
df.drop(columns=['col1', 'col2'])

# ── Filter rows ───────────────────────────────────────────────────────────────
df[df['age'] > 40]
df[(df['age'] > 40) & (df['overtime'] == 1)]  # AND
df[(df['dept'] == 'HR') | (df['dept'] == 'R&D')]  # OR

# ── Select numeric / categorical columns automatically ───────────────────────
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

# ── Separate X and y ─────────────────────────────────────────────────────────
y = df['attrition']
X = df.drop(columns=['attrition'])


---
## 4 · GroupBy and Aggregation


In [ ]:
# ── Group means ──────────────────────────────────────────────────────────────
df.groupby('attrition')[['monthly_income', 'age']].mean()
df.groupby('attrition').mean(numeric_only=True)  # all numeric cols

# ── Multiple aggregations ─────────────────────────────────────────────────────
df.groupby('department').agg({
    'monthly_income': ['mean', 'median', 'std'],
    'attrition': 'mean'   # attrition rate per department
})

# ── Cross-tabulation (for two categorical variables) ─────────────────────────
pd.crosstab(df['overtime'], df['attrition'])
pd.crosstab(df['overtime'], df['attrition'], normalize='index')  # row proportions

# ── Pivot table ───────────────────────────────────────────────────────────────
df.pivot_table(values='attrition', index='department',
               columns='overtime', aggfunc='mean')

# ── Correlation matrix ────────────────────────────────────────────────────────
df.corr(numeric_only=True)
df.corr(numeric_only=True)['attrition'].sort_values()  # correlation with target


---
## 5 · Distributions — Histograms and Density Plots


In [ ]:
# ── Single histogram ─────────────────────────────────────────────────────────
plt.hist(df['monthly_income'], bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Monthly Income'); plt.ylabel('Count')
plt.title('Distribution of Monthly Income'); plt.tight_layout(); plt.show()

# ── Histogram split by group ─────────────────────────────────────────────────
for group, sub in df.groupby('attrition'):
    plt.hist(sub['monthly_income'], bins=25, alpha=0.6,
             label=f'Attrition={group}', density=True)
plt.xlabel('Monthly Income'); plt.legend(); plt.title('Income by Attrition')
plt.tight_layout(); plt.show()

# ── KDE (density) plot — seaborn ─────────────────────────────────────────────
sns.kdeplot(data=df, x='monthly_income', hue='attrition', fill=True, alpha=0.4)
plt.title('Income Density by Attrition'); plt.tight_layout(); plt.show()

# ── Histogram + KDE together ─────────────────────────────────────────────────
sns.histplot(df['age'], kde=True, bins=25, color='teal')
plt.title('Age Distribution'); plt.tight_layout(); plt.show()


---
## 6 · Boxplots and Violin Plots


In [ ]:
# ── Single boxplot ───────────────────────────────────────────────────────────
sns.boxplot(x='attrition', y='monthly_income', data=df)
plt.title('Income by Attrition'); plt.tight_layout(); plt.show()

# ── Multiple boxplots in subplots ─────────────────────────────────────────────
num_cols = ['monthly_income', 'age', 'years_at_company', 'distance_from_home']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(x='attrition', y=col, data=df, ax=ax)
    ax.set_title(col)
plt.suptitle('Numeric Variables by Attrition', y=1.01)
plt.tight_layout(); plt.show()

# ── Violin plot (shows distribution shape, not just quartiles) ───────────────
sns.violinplot(x='attrition', y='monthly_income', data=df, inner='box')
plt.title('Income Distribution by Attrition'); plt.tight_layout(); plt.show()


---
## 7 · Bar Charts (for Categorical Variables)


In [ ]:
# ── Attrition rate by categorical variable ────────────────────────────────────
sns.barplot(x='department', y='attrition', data=df, estimator='mean',
            errorbar='ci')   # shows 95% CI bars
plt.ylabel('Attrition Rate'); plt.title('Attrition Rate by Department')
plt.tight_layout(); plt.show()

# ── Count plot (raw counts, coloured by group) ───────────────────────────────
sns.countplot(x='department', hue='attrition', data=df)
plt.title('Counts by Department and Attrition')
plt.tight_layout(); plt.show()

# ── Horizontal bar chart (good for many categories or long labels) ───────────
df['department'].value_counts().plot(kind='barh', color='steelblue')
plt.xlabel('Count'); plt.title('Employees per Department')
plt.tight_layout(); plt.show()

# ── Proportion bar chart using crosstab ──────────────────────────────────────
ct = pd.crosstab(df['department'], df['attrition'], normalize='index')
ct.plot(kind='bar', stacked=True, colormap='RdYlGn')
plt.ylabel('Proportion'); plt.title('Attrition Proportion by Department')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()


---
## 8 · Scatter Plots


In [ ]:
# ── Basic scatter ─────────────────────────────────────────────────────────────
plt.scatter(df['age'], df['monthly_income'], alpha=0.4, s=20)
plt.xlabel('Age'); plt.ylabel('Monthly Income')
plt.title('Age vs Income'); plt.tight_layout(); plt.show()

# ── Scatter coloured by group ─────────────────────────────────────────────────
colors = {0: 'steelblue', 1: 'tomato'}
for val, group in df.groupby('attrition'):
    plt.scatter(group['age'], group['monthly_income'],
                label=f'Attrition={val}', alpha=0.5, s=20,
                color=colors[val])
plt.xlabel('Age'); plt.ylabel('Monthly Income'); plt.legend()
plt.title('Age vs Income by Attrition'); plt.tight_layout(); plt.show()

# ── seaborn scatter with regression line ─────────────────────────────────────
sns.regplot(x='age', y='monthly_income', data=df, scatter_kws={'alpha':0.3})
plt.title('Age vs Income with Regression Line'); plt.tight_layout(); plt.show()

# ── Scatter with hue (seaborn) ───────────────────────────────────────────────
sns.scatterplot(x='age', y='monthly_income', hue='attrition',
                data=df, alpha=0.6, s=40)
plt.title('Age vs Income by Attrition'); plt.tight_layout(); plt.show()


---
## 9 · Correlation Heatmap and Pairplot


In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr,
            annot=True,       # show correlation values
            fmt='.2f',        # 2 decimal places
            cmap='RdBu_r',    # red = positive, blue = negative
            center=0,
            square=True,
            linewidths=0.5)
plt.title('Correlation Matrix'); plt.tight_layout(); plt.show()

# ── Correlation with target only (quick overview) ────────────────────────────
target_corr = df.corr(numeric_only=True)['attrition'].drop('attrition').sort_values()
target_corr.plot(kind='barh', figsize=(7, 5), color=target_corr.map(
    lambda x: 'tomato' if x > 0 else 'steelblue'))
plt.axvline(0, color='black', lw=0.8)
plt.title('Correlation with Attrition'); plt.tight_layout(); plt.show()

# ── Pairplot (scatterplot matrix — slow for many columns, use max ~6 cols) ───
cols = ['monthly_income', 'age', 'years_at_company', 'attrition']
sns.pairplot(df[cols], hue='attrition', diag_kind='kde',
             plot_kws={'alpha': 0.4, 's': 20})
plt.suptitle('Pairplot', y=1.02); plt.show()


---
## 10 · Line Plots, Fitted Curves, and Residual Plots


In [ ]:
# ── Line plot (e.g. CV-MSE vs K, or loss curve) ─────────────────────────────
K_vals = range(1, 11)
mse_vals = [10, 7, 5.5, 4.8, 4.9, 5.1, 5.3, 5.6, 5.9, 6.2]  # example

plt.plot(K_vals, mse_vals, 'o-', color='steelblue', ms=6)
plt.axvline(5, ls='--', color='tomato', label='Optimal K=5')
plt.xlabel('K'); plt.ylabel('CV-MSE')
plt.title('Cross-Validation Error vs K'); plt.legend()
plt.tight_layout(); plt.show()

# ── Scatter + fitted curve ────────────────────────────────────────────────────
x = np.linspace(-3, 3, 80)
y = 0.8*x**3 - 2*x + np.random.normal(0, 1, 80)
x_grid = np.linspace(-3, 3, 200)

plt.scatter(x, y, alpha=0.5, s=20, label='Data')
plt.plot(x_grid, 0.8*x_grid**3 - 2*x_grid, color='red', lw=2, label='True curve')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.title('Scatter with Fitted Curve'); plt.tight_layout(); plt.show()

# ── Residual plot (after fitting a model) ────────────────────────────────────
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(x.reshape(-1,1), y)
y_hat = model.predict(x.reshape(-1,1))
residuals = y - y_hat

plt.scatter(y_hat, residuals, alpha=0.5, s=20, color='steelblue')
plt.axhline(0, color='red', lw=1, ls='--')
plt.xlabel('Fitted values'); plt.ylabel('Residuals')
plt.title('Residual Plot'); plt.tight_layout(); plt.show()


---
## 11 · Subplots Grid (Multiple Plots in One Figure)


In [ ]:
# ── 2x2 grid ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()   # makes it easy to index: axes[0], axes[1], ...

for i, col in enumerate(['age', 'monthly_income', 'years_at_company', 'distance_from_home']):
    axes[i].hist(df[col], bins=25, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel(col)

plt.suptitle('Distributions of Numeric Variables', fontsize=13)
plt.tight_layout(); plt.show()

# ── Row of plots (1xN) ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(x='attrition', y='monthly_income', data=df, ax=axes[0])
sns.boxplot(x='attrition', y='age',            data=df, ax=axes[1])
sns.countplot(x='overtime', hue='attrition',   data=df, ax=axes[2])

axes[0].set_title('Income')
axes[1].set_title('Age')
axes[2].set_title('Overtime')

plt.suptitle('Key Predictors vs Attrition', fontsize=13)
plt.tight_layout(); plt.show()


---
## 12 · Model Evaluation — Metrics and Plots


In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve,
                              mean_squared_error, r2_score)

# ── Confusion matrix as heatmap ───────────────────────────────────────────────
def plot_confusion(y_true, y_pred, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.title(title); plt.tight_layout(); plt.show()

# plot_confusion(y_te, y_pred)

# ── ROC Curve ─────────────────────────────────────────────────────────────────
def plot_roc(y_true, y_prob, label='Model'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})')
    plt.plot([0,1],[0,1], '--', color='grey')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title('ROC Curve'); plt.legend()
    plt.tight_layout(); plt.show()

# ── Multiple ROC curves on same plot ─────────────────────────────────────────
def plot_roc_multi(models_dict, y_true):
    # models_dict = {'LR': y_prob_lr, 'RF': y_prob_rf, ...}
    for name, y_prob in models_dict.items():
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    plt.plot([0,1],[0,1], '--', color='grey', label='Random')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title('ROC Curve Comparison'); plt.legend()
    plt.tight_layout(); plt.show()

# ── Classification report (full table) ───────────────────────────────────────
# print(classification_report(y_te, y_pred, target_names=['No', 'Yes']))

# ── Regression metrics ────────────────────────────────────────────────────────
# mse  = mean_squared_error(y_te, y_pred)
# rmse = np.sqrt(mse)
# r2   = r2_score(y_te, y_pred)
# print(f"RMSE: {rmse:.4f}  |  R²: {r2:.4f}")


---
## 13 · Bootstrap Result Plot


In [ ]:
# ── Standard bootstrap histogram with CI ─────────────────────────────────────
def plot_bootstrap(boot_samples, observed, stat_name='Statistic'):
    ci = np.percentile(boot_samples, [2.5, 97.5])
    plt.figure(figsize=(7, 4))
    plt.hist(boot_samples, bins=50, color='steelblue',
             edgecolor='white', alpha=0.8, label='Bootstrap distribution')
    plt.axvline(observed, color='red', lw=2, label=f'Observed = {observed:.3f}')
    plt.axvline(ci[0], color='orange', ls='--', label=f'95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]')
    plt.axvline(ci[1], color='orange', ls='--')
    plt.xlabel(stat_name); plt.ylabel('Frequency')
    plt.title(f'Bootstrap Sampling Distribution of {stat_name}')
    plt.legend(); plt.tight_layout(); plt.show()
    print(f"Bootstrap SE : {np.std(boot_samples):.4f}")
    print(f"95% CI       : [{ci[0]:.4f}, {ci[1]:.4f}]")
    return ci


---
## 14 · Model Comparison Tables


In [ ]:
# ── Build a comparison table for multiple models ──────────────────────────────
from sklearn.metrics import accuracy_score, roc_auc_score

def compare_models(models_dict, X_te, y_te, threshold=0.5):
    '''
    models_dict = {'LR': lr_model, 'RF': rf_model, ...}
    All models must support predict_proba.
    '''
    rows = []
    for name, model in models_dict.items():
        y_prob = model.predict_proba(X_te)[:, 1]
        y_pred = (y_prob >= threshold).astype(int)
        rows.append({
            'Model'   : name,
            'Accuracy': round(accuracy_score(y_te, y_pred), 4),
            'AUC'     : round(roc_auc_score(y_te, y_prob), 4),
        })
    return pd.DataFrame(rows).sort_values('AUC', ascending=False)

# Usage:
# result = compare_models({'LR': lr, 'Tree': tree, 'RF': rf, 'GBM': gbm}, X_te, y_te)
# print(result)

# ── Manual comparison table ───────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model'    : ['Logistic Regression', 'Decision Tree', 'Random Forest', 'GBM'],
    'Test Acc' : [0.8778, 0.8722, 0.8778, 0.8800],
    'Test AUC' : [0.7066, 0.6883, 0.6900, 0.7100],
})
print(summary.to_string(index=False))


---
## 15 · Custom Classification Threshold


In [ ]:
# ── Try multiple thresholds and see effect on confusion matrix ────────────────
from sklearn.metrics import confusion_matrix, classification_report

def threshold_analysis(y_true, y_prob, thresholds=[0.2, 0.3, 0.4, 0.5]):
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        print(f"Threshold {t:.2f} | TP={tp} FP={fp} FN={fn} TN={tn} "
              f"| Recall={recall:.2f} Precision={precision:.2f}")

# Usage: threshold_analysis(y_te, y_prob, [0.2, 0.3, 0.4, 0.5])

# ── Optimal threshold from cost perspective ───────────────────────────────────
# t* = c_FP / (c_FP + c_FN)
# If FN costs 5x more than FP: t* = 1/(1+5) = 0.167
c_FP, c_FN = 1, 5
t_star = c_FP / (c_FP + c_FN)
print(f"Cost-optimal threshold: {t_star:.3f}")
# Source: Compendium Ch 4 (Classification, cost-sensitive threshold)


---
## 16 · Variable Importance Plot (Trees / RF / GBM)


In [ ]:
def plot_importance(model, feature_names, title='Variable Importance', top_n=15):
    imp = pd.Series(model.feature_importances_, index=feature_names)
    imp = imp.sort_values(ascending=True).tail(top_n)   # top N variables
    
    plt.figure(figsize=(7, max(4, top_n * 0.35)))
    imp.plot(kind='barh', color='teal', edgecolor='white')
    plt.xlabel('Importance (Mean Decrease in Gini)')
    plt.title(title)
    plt.tight_layout(); plt.show()

# Usage:
# plot_importance(rf, X_tr.columns, 'Random Forest — Variable Importance')
# plot_importance(gbm, X_tr.columns, 'GBM — Variable Importance')


---
## 17 · Quick Reference — Most-Used One-Liners

```python
# ── Load & inspect ────────────────────────────────────────────────────────────
df = pd.read_csv("file.csv")
df.shape, df.dtypes, df.isnull().sum(), df.describe()

# ── Recode ────────────────────────────────────────────────────────────────────
df['y'] = df['y'].map({'Yes': 1, 'No': 0})
df = pd.get_dummies(df, columns=['dept'], drop_first=True)

# ── Group stats ───────────────────────────────────────────────────────────────
df.groupby('y').mean(numeric_only=True)
pd.crosstab(df['cat_col'], df['y'], normalize='index')
df.corr(numeric_only=True)['y'].sort_values()

# ── Split ─────────────────────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            random_state=42, stratify=y)

# ── Evaluate ──────────────────────────────────────────────────────────────────
accuracy_score(y_te, y_pred)
roc_auc_score(y_te, y_prob)
confusion_matrix(y_te, y_pred)
classification_report(y_te, y_pred)
mean_squared_error(y_te, y_pred, squared=False)   # RMSE
r2_score(y_te, y_pred)
```
